In [27]:
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
import sys
import os

# Para importar build_preprocessor desde src/
sys.path.append(os.path.abspath(".."))
from src.preprocessing import build_preprocessor

# ============================
# 1. Cargar dataset procesado
# ============================
df = pd.read_csv("../data/processed/04_default_credit_features.csv")

# Eliminar ID
df = df.drop(columns=["ID"])

TARGET = "default payment next month"

# ============================
# 2. Convertir booleanos a string
# ============================
bool_cols = df.select_dtypes(include=["bool"]).columns
df[bool_cols] = df[bool_cols].astype("string")

# ============================
# 3. Separar X e y
# ============================
X = df.drop(columns=[TARGET])
y = df[TARGET]

# ============================
# 4. Detectar columnas numéricas y categóricas
# ============================
cat_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

print("Categóricas:", cat_cols)
print("Numéricas:", num_cols)

# ============================
# 5. Train/Test Split
# ============================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ============================
# 6. Construir preprocesador
# ============================
preprocessor = build_preprocessor(num_cols, cat_cols)

# ============================
# 7. Pipeline final
# ============================
pipeline = Pipeline(steps=[
    ("preprocessing", preprocessor),
    ("model", LogisticRegression(max_iter=5000))
])

# ============================
# 8. Entrenar pipeline
# ============================
pipeline.fit(X_train, y_train)
print("✓ Pipeline entrenado correctamente")

# ============================
# 9. Guardar preprocesador
# ============================
joblib.dump(preprocessor, "../models/preprocessor.pkl")
print("✓ preprocessor.pkl guardado")

# ============================
# 10. Guardar pipeline completo
# ============================
joblib.dump(pipeline, "../models/pipeline_credit.joblib")
print("✓ pipeline_credit.joblib guardado")

Categóricas: ['SEX_2', 'EDUCATION_1', 'EDUCATION_2', 'EDUCATION_3', 'EDUCATION_4', 'EDUCATION_5', 'EDUCATION_6', 'MARRIAGE_1', 'MARRIAGE_2', 'MARRIAGE_3', 'age_group', 'credit_level']
Numéricas: ['LIMIT_BAL', 'AGE', 'PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'avg_bill', 'credit_utilization', 'avg_pay', 'payment_ratio', 'late_payments', 'bill_trend', 'pay_trend', 'risk_score', 'ability_to_pay', 'log_limit', 'log_avg_bill']
✓ Pipeline entrenado correctamente
✓ preprocessor.pkl guardado
✓ pipeline_credit.joblib guardado
